# EN3160 Assignment 03: Neural Networks
**Karunarathna G.K.T. | 230322U**  
Repository: https://github.com/KavijaK/EN3160_A03  
GitHub profile: https://github.com/KavijaK


In [ ]:
from pathlib import Path
import json
import random
import time
import urllib.request
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

SEED = 230322
RETRAIN = False  # Set True to repeat completed training runs.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(4)
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
data_dir, results, checkpoints = map(Path, ["data", "results", "checkpoints"])
for folder in [data_dir, results, checkpoints]:
    folder.mkdir(exist_ok=True)
torch.hub.set_dir(str(data_dir / "torch_hub"))
print(f"PyTorch {torch.__version__}; device: {device}; seed: {SEED}")


In [ ]:
def make_loaders(train, train_eval, evaluation, batch_size):
    generator = torch.Generator().manual_seed(SEED)
    return {
        "train": DataLoader(train, batch_size, shuffle=True, generator=generator),
        "train_eval": DataLoader(train_eval, batch_size, shuffle=False),
        "eval": DataLoader(evaluation, batch_size, shuffle=False),
    }

@torch.inference_mode()
def evaluate(predict, loader, classes):
    loss, correct, count = 0., 0, 0
    matrix = torch.zeros(classes, classes, dtype=torch.int64)
    for images, labels in loader:
        logits = predict(images.to(device))
        target = labels.to(device)
        prediction = logits.argmax(1)
        loss += F.cross_entropy(logits, target, reduction="sum").item()
        correct += (prediction == target).sum().item()
        count += len(labels)
        matrix += torch.bincount(labels*classes + prediction.cpu(),
                                 minlength=classes**2).reshape(classes, classes)
    return {"loss": loss/count, "accuracy": 100*correct/count}, matrix.numpy()

def fit(model, loaders, name, epochs, optimizer, classes, scheduler=None, frozen=False):
    weights, log = checkpoints / f"{name}.pt", results / f"{name}_history.json"
    if not RETRAIN and weights.exists() and log.exists():
        history = json.loads(log.read_text())
        if len(history) == epochs:
            model.load_state_dict(torch.load(weights, map_location=device, weights_only=True))
            print(f"{name}: loaded completed {epochs}-epoch run")
            return history
    history = []
    started = time.perf_counter()
    for epoch in range(1, epochs+1):
        model.train(not frozen)  # Frozen backbone keeps its pretrained BN statistics.
        for images, labels in loaders["train"]:
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(images.to(device)), labels.to(device))
            loss.backward()
            optimizer.step()
        model.eval()
        train, _ = evaluate(model, loaders["train_eval"], classes)
        heldout, _ = evaluate(model, loaders["eval"], classes)
        row = dict(epoch=epoch, train_loss=train["loss"], train_accuracy=train["accuracy"],
                   eval_loss=heldout["loss"], eval_accuracy=heldout["accuracy"],
                   elapsed_seconds=time.perf_counter()-started)
        history.append(row)
        log.write_text(json.dumps(history, indent=2))
        print(f"{name} {epoch:02d}/{epochs}: train {train['accuracy']:.2f}%, "
              f"evaluation {heldout['accuracy']:.2f}%", flush=True)
        if scheduler is not None:
            scheduler.step()
    torch.save(model.state_dict(), weights)
    return history

def plot_history(histories, filename, evaluation_label="Test"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, history in histories.items():
        epochs = [r["epoch"] for r in history]
        for ax, metric in zip(axes, ["accuracy", "loss"]):
            line, = ax.plot(epochs, [r[f"train_{metric}"] for r in history], label=f"{name}: train")
            ax.plot(epochs, [r[f"eval_{metric}"] for r in history], "--",
                    color=line.get_color(), label=f"{name}: {evaluation_label.lower()}")
            ax.set(xlabel="Epoch", ylabel="Accuracy (%)" if metric=="accuracy" else "Cross-entropy loss")
            ax.grid(alpha=.2)
            ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(results / filename, dpi=160, bbox_inches="tight")
    plt.show()

def plot_confusions(matrices, labels, filename):
    fig, axes = plt.subplots(1, len(matrices), figsize=(6*len(matrices), 5), squeeze=False)
    for ax, (name, matrix) in zip(axes.flat, matrices.items()):
        ax.imshow(matrix, cmap="Blues")
        ax.set(xticks=range(len(labels)), yticks=range(len(labels)),
               xticklabels=labels, yticklabels=labels, xlabel="Predicted", ylabel="True", title=name)
        for (row, col), value in np.ndenumerate(matrix):
            ax.text(col, row, str(value), ha="center", va="center", fontsize=7,
                    color="white" if value > matrix.max()/2 else "black")
    fig.tight_layout()
    fig.savefig(results / filename, dpi=160, bbox_inches="tight")
    plt.show()
